# Research2.csv 생성 노트북
2014-2025 기간, Bitcoin,VIX 추가, QQQ로 `research2.csv`를 생성합니다.

In [16]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path(r'c:/Users/a00548169/OneDrive - ONEVIRTUALOFFICE/Desktop/자기주도적/ETFpricepredictmodel/data/2014-2025/USD')

In [17]:
# 유틸리티 함수: 공통 CSV 로딩 & 정제
import re

def _detect_cols(df, date_col, price_col):
    cols_lower = {c: str(c).strip().lower() for c in df.columns}
    # 날짜 컬럼 탐색
    date_candidates = [
        date_col,
        'Date', 'date', '날짜'
    ]
    detected_date = None
    for cand in date_candidates:
        for c, lc in cols_lower.items():
            if lc == str(cand).lower():
                detected_date = c
                break
        if detected_date is not None:
            break
    if detected_date is None:
        # 첫 번째 datetime으로 파싱 가능한 컬럼 시도
        for c in df.columns:
            try:
                pd.to_datetime(df[c], errors='raise')
                detected_date = c
                break
            except Exception:
                continue
        if detected_date is None:
            detected_date = df.columns[0]
    # 가격 컬럼 탐색
    price_candidates = [
        price_col,
        'Price', 'price', '종가', 'close', 'Close'
    ]
    detected_price = None
    for cand in price_candidates:
        for c, lc in cols_lower.items():
            if lc == str(cand).lower():
                detected_price = c
                break
        if detected_price is not None:
            break
    if detected_price is None:
        # 수치형 컬럼 중 가장 적합한 것 선택
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if numeric_cols:
            detected_price = numeric_cols[0]
        else:
            # 문자열이라도 가격처럼 보이는 첫 컬럼 선택
            detected_price = df.columns[1] if len(df.columns) > 1 else df.columns[0]
    return detected_date, detected_price


def read_price_csv(filepath, date_col='Date', price_col='Price', new_name=None):
    df = pd.read_csv(filepath, encoding='utf-8-sig')
    # 컬럼 자동 탐지
    date_col_detected, price_col_detected = _detect_cols(df, date_col, price_col)
    # 날짜 파싱 (다양한 포맷 허용)
    df[date_col_detected] = pd.to_datetime(df[date_col_detected], errors='coerce')
    # 가격 숫자화 (천단위 콤마/퍼센트 제거)
    if price_col_detected in df.columns:
        df[price_col_detected] = (df[price_col_detected].astype(str)
                                  .str.replace(',', '', regex=False)
                                  .str.replace('%', '', regex=False))
        df[price_col_detected] = pd.to_numeric(df[price_col_detected], errors='coerce')
    # 필요 컬럼만 남기기
    keep = [date_col_detected] + ([price_col_detected] if price_col_detected in df.columns else [])
    df = df[keep].dropna(subset=[date_col_detected]).copy()
    if new_name and (price_col_detected in df.columns):
        df.rename(columns={date_col_detected: 'Date', price_col_detected: new_name}, inplace=True)
    else:
        df.rename(columns={date_col_detected: 'Date'}, inplace=True)
    df = df.drop_duplicates('Date').sort_values('Date').reset_index(drop=True)
    return df

In [18]:
# 뉴스: 일별 헤드라인 선택 (경제 키워드 우선, 없으면 렉시콘 가산점)
news_path = BASE_DIR / 'US_news.csv'
news_raw = pd.read_csv(news_path, encoding='utf-8-sig')

# 날짜 컬럼 탐지
date_col = 'Date' if 'Date' in news_raw.columns else ('date' if 'date' in news_raw.columns else None)
if date_col is None:
    raise ValueError('US_news.csv에 날짜 컬럼(Date 또는 date)이 필요합니다.')
news_raw[date_col] = pd.to_datetime(news_raw[date_col], errors='coerce')

# 헤드라인 컬럼 탐지
hl_col = 'headline' if 'headline' in news_raw.columns else None
if hl_col is None:
    text_candidates = [c for c in news_raw.columns if c != date_col and news_raw[c].dtype == object]
    if text_candidates:
        hl_col = text_candidates[0]
    else:
        raise ValueError('US_news.csv에서 헤드라인 텍스트 컬럼을 찾을 수 없습니다.')

# 경제 키워드 세트
import re

equity_market = [
    'stock','stocks','equity','equities','share','shares',
    'market','markets','benchmark','index','indices',
    's&p','s&p 500','nasdaq','dow','djia','russell',
    'etf','etfs','mutual fund','hedge fund',
    'ipo','secondary offering','listing','delisting',
    'volatility','vix','liquidity','trading volume',
    'bull market','bear market','correction','rally','selloff'
]
corporate_finance = [
    'earnings','revenue','sales','profit','loss','net income',
    'operating income','ebit','ebitda','margin',
    'guidance','outlook','forecast',
    'quarter','quarterly','annual','fiscal',
    'balance sheet','income statement','cash flow',
    'dividend','payout','buyback','share repurchase',
    'valuation','p/e','price to earnings','market cap'
]
macro_economy = [
    'economy','economic','macroeconomic',
    'gdp','gross domestic product',
    'inflation','deflation','cpi','ppi','core inflation',
    'unemployment','employment','jobs','jobless',
    'payrolls','labor market','wages',
    'consumer spending','retail sales',
    'manufacturing','ism','pmi',
    'recession','slowdown','recovery','growth'
]
monetary_policy = [
    'fed','federal reserve','fomc',
    'central bank','ecb','boj','boe','pboc',
    'interest rate','rates','policy rate',
    'rate hike','rate cut','tightening','easing',
    'quantitative easing','qe',
    'quantitative tightening','qt',
    'monetary policy','forward guidance'
]
fixed_income = [
    'bond','bonds','treasury','treasuries',
    'yield','yield curve','inversion',
    'credit','credit market','credit risk',
    'spread','yield spread',
    'investment grade','high yield','junk bond',
    'default','downgrade','rating','moody','s&p rating'
]
banking = [
    'bank','banks','banking sector',
    'loan','lending','mortgage',
    'deposit','liquidity',
    'capital adequacy','stress test',
    'financial stability','systemic risk',
    'bank failure','bailout'
]
fx_global = [
    'currency','currencies','forex','fx',
    'exchange rate','devaluation','appreciation',
    'dollar','usd','euro','eur','yen','jpy',
    'yuan','cny','pound','gbp',
    'trade balance','current account','capital flow'
]

econ_terms = [t for t in (equity_market + corporate_finance + macro_economy + monetary_policy + fixed_income + banking + fx_global) if t]

# Economic_Lexicon.csv 토큰 로드
lexicon_path = BASE_DIR / 'Economic_Lexicon.csv'
lexicon_terms = []
if lexicon_path.exists():
    try:
        lex_df = pd.read_csv(lexicon_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        lex_df = pd.read_csv(lexicon_path, encoding='cp949')
    token_col = 'token' if 'token' in lex_df.columns else lex_df.columns[0]
    lexicon_terms = (lex_df[token_col].dropna().astype(str).str.strip().unique().tolist())
    # 너무 짧은 토큰 제외
    lexicon_terms = [t for t in lexicon_terms if t and len(t) >= 2]

# 중복 제거 + 긴 용어 우선

def _dedup_sort(terms):
    seen = set(); out = []
    for t in terms:
        tl = t.lower()
        if tl not in seen:
            seen.add(tl); out.append(t)
    out.sort(key=len, reverse=True)
    return out

econ_terms = _dedup_sort(econ_terms)
lexicon_terms = _dedup_sort(lexicon_terms)

# FlashText 사용 가능 여부
try:
    from flashtext import KeywordProcessor
    _USE_FLASHTEXT = True
except Exception:
    _USE_FLASHTEXT = False

if _USE_FLASHTEXT:
    econ_kp = KeywordProcessor(case_sensitive=False)
    for term in econ_terms:
        econ_kp.add_keyword(term)
    lex_kp = KeywordProcessor(case_sensitive=False)
    for term in lexicon_terms:
        lex_kp.add_keyword(term)

    def econ_score(text: str) -> int:
        if not isinstance(text, str):
            text = str(text)
        return len(set(econ_kp.extract_keywords(text)))
    def lex_score(text: str) -> int:
        if not isinstance(text, str):
            text = str(text)
        return len(set(lex_kp.extract_keywords(text)))
else:
    econ_pat = re.compile('(?:' + '|'.join([re.escape(t) for t in econ_terms]) + ')', flags=re.IGNORECASE) if econ_terms else None
    lex_pat  = re.compile('(?:' + '|'.join([re.escape(t) for t in lexicon_terms]) + ')', flags=re.IGNORECASE) if lexicon_terms else None

    def econ_score(text: str) -> int:
        if econ_pat is None:
            return 0
        if not isinstance(text, str):
            text = str(text)
        return len({m.group(0).lower() for m in econ_pat.finditer(text)})
    def lex_score(text: str) -> int:
        if lex_pat is None:
            return 0
        if not isinstance(text, str):
            text = str(text)
        return len({m.group(0).lower() for m in lex_pat.finditer(text)})

# 점수 계산 및 날짜별 선택: 경제 키워드 우선, 없으면 렉시콘 점수
nr = news_raw.dropna(subset=[date_col, hl_col]).copy()
nr['__text'] = nr[hl_col].astype(str)
nr['__econ'] = nr['__text'].apply(econ_score)
nr['__lex']  = nr['__text'].apply(lex_score)
nr['__len']  = nr['__text'].str.len()
nr['__has_econ'] = nr['__econ'] > 0

# 정렬: 날짜↑, 경제키워드 여부(True) 우선, 경제점수↓, 렉시콘점수↓, 길이↓
nr.sort_values([date_col, '__has_econ', '__econ', '__lex', '__len'], ascending=[True, False, False, False, False], inplace=True)

headlines = nr.drop_duplicates(subset=[date_col])[[date_col, hl_col]].rename(columns={date_col: 'Date', hl_col: 'Headline'})
headlines = headlines.drop_duplicates('Date').sort_values('Date').reset_index(drop=True)
headlines.head()

,Date,Headline
0,2007-07-07,Rating the U.S. Players at the Copa
1,2014-01-01,"As Latvia Adopts Euro, Future Growth Is Slowing"
2,2014-01-02,Russell Banks: By the Book
3,2014-01-03,Annual Sales for American Automakers Reach a 6...
4,2014-01-04,"Help the Working Poor, but Share the Burden"


In [19]:
# 금, 원유, 채권
gold = read_price_csv(BASE_DIR / 'US_gold.csv', date_col='Date', price_col='Price', new_name='gold')
oil = read_price_csv(BASE_DIR / 'US_oil.csv', date_col='Date', price_col='Price', new_name='oil')
bond = read_price_csv(BASE_DIR / 'US_bond.csv', date_col='Date', price_col='Price', new_name='bond')
gold['gold'] = gold['gold'].round(2)
oil['oil'] = oil['oil'].round(2)
bond['bond'] = bond['bond'].round(3)
gold, oil, bond

(           Date    gold
 0    2014-01-02  1225.2
 1    2014-01-03  1238.6
 2    2014-01-06  1238.0
 3    2014-01-07  1229.6
 4    2014-01-08  1225.5
 ...         ...     ...
 3070 2025-12-25  4527.5
 3071 2025-12-26  4552.7
 3072 2025-12-29  4327.0
 3073 2025-12-30  4386.3
 3074 2025-12-31  4341.1
 
 [3075 rows x 2 columns],
            Date    oil
 0    2014-01-01  98.70
 1    2014-01-02  95.44
 2    2014-01-03  93.96
 3    2014-01-06  93.43
 4    2014-01-07  93.67
 ...         ...    ...
 3149 2025-12-25  58.53
 3150 2025-12-26  56.74
 3151 2025-12-29  58.08
 3152 2025-12-30  57.95
 3153 2025-12-31  57.42
 
 [3154 rows x 2 columns],
            Date   bond
 0    2014-01-01  3.026
 1    2014-01-02  2.991
 2    2014-01-03  2.998
 3    2014-01-06  2.958
 4    2014-01-07  2.941
 ...         ...    ...
 3125 2025-12-25  4.133
 3126 2025-12-26  4.134
 3127 2025-12-29  4.116
 3128 2025-12-30  4.128
 3129 2025-12-31  4.153
 
 [3130 rows x 2 columns])

In [20]:
# 환율: USD_CAD, USD_CNY, USD_EUR, USD_JPY, USD_MXN
cad = read_price_csv(BASE_DIR / 'USD_CAD.csv', new_name='cad')
cny = read_price_csv(BASE_DIR / 'USD_CNY.csv', new_name='cny')
eur = read_price_csv(BASE_DIR / 'USD_EUR.csv', new_name='eur')
jpy = read_price_csv(BASE_DIR / 'USD_JPY.csv', new_name='jpy')
mxn = read_price_csv(BASE_DIR / 'USD_MXN.csv', new_name='mxn')
cad, cny, eur, jpy, mxn

(           Date     cad
 0    2014-01-01  1.0641
 1    2014-01-02  1.0668
 2    2014-01-03  1.0632
 3    2014-01-06  1.0653
 4    2014-01-07  1.0764
 ...         ...     ...
 3126 2025-12-25  1.3682
 3127 2025-12-26  1.3672
 3128 2025-12-29  1.3691
 3129 2025-12-30  1.3697
 3130 2025-12-31  1.3725
 
 [3131 rows x 2 columns],
            Date     cny
 0    2014-01-01  6.0540
 1    2014-01-02  6.0507
 2    2014-01-03  6.0515
 3    2014-01-06  6.0527
 4    2014-01-07  6.0512
 ...         ...     ...
 3127 2025-12-25  7.0059
 3128 2025-12-26  7.0067
 3129 2025-12-29  7.0060
 3130 2025-12-30  6.9964
 3131 2025-12-31  6.9937
 
 [3132 rows x 2 columns],
            Date     eur
 0    2014-01-01  0.7271
 1    2014-01-02  0.7314
 2    2014-01-03  0.7359
 3    2014-01-06  0.7338
 4    2014-01-07  0.7345
 ...         ...     ...
 3126 2025-12-25  0.8490
 3127 2025-12-26  0.8494
 3128 2025-12-29  0.8495
 3129 2025-12-30  0.8513
 3130 2025-12-31  0.8514
 
 [3131 rows x 2 columns],
            Date

In [21]:
# VIX: S&P 500 Volatility Index
vix = read_price_csv(BASE_DIR / 'US_VIX.csv', date_col='Date', price_col='Price', new_name='vix')
vix['vix'] = vix['vix'].round(3)
vix.head()

,Date,vix
0,2014-01-02,14.23
1,2014-01-03,13.76
2,2014-01-06,13.55
3,2014-01-07,12.92
4,2014-01-08,12.87


In [22]:
# ETF: QQQ (로컬 CSV 사용)
qqq = read_price_csv(BASE_DIR / 'qqq.csv', date_col='Date', price_col='Price', new_name='ETF')
qqq['ETF'] = qqq['ETF'].round(2)
qqq.head()

,Date,ETF
0,2014-01-02,86.86
1,2014-01-03,86.24
2,2014-01-06,85.92
3,2014-01-07,86.71
4,2014-01-08,86.90


In [23]:
# 비트코인: 로컬 CSV (한국어 컬럼 처리)
btc_path = BASE_DIR / 'Bitcoin.csv'
try:
    btc_raw = pd.read_csv(btc_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    btc_raw = pd.read_csv(btc_path, encoding='cp949')
# 예상 컬럼: 날짜, 종가
date_col = '날짜' if '날짜' in btc_raw.columns else ('Date' if 'Date' in btc_raw.columns else None)
price_col = '종가' if '종가' in btc_raw.columns else ('Price' if 'Price' in btc_raw.columns else None)
if date_col is None or price_col is None:
    raise ValueError('Bitcoin.csv에 날짜/종가 컬럼을 찾을 수 없습니다.')
btc_raw[date_col] = pd.to_datetime(btc_raw[date_col], errors='coerce')
btc_raw[price_col] = btc_raw[price_col].astype(str).str.replace(',', '', regex=False)
btc_raw[price_col] = pd.to_numeric(btc_raw[price_col], errors='coerce')
bitcoin = btc_raw[[date_col, price_col]].rename(columns={date_col: 'Date', price_col: 'bitcoin'})
bitcoin = bitcoin.drop_duplicates('Date').sort_values('Date').reset_index(drop=True)
bitcoin.head()

,Date,bitcoin
0,2014-01-01,815.9
1,2014-01-02,856.9
2,2014-01-03,884.3
3,2014-01-04,924.7
4,2014-01-05,1014.7


In [24]:
# 날짜 범위 생성: 2014-01-01 ~ 2025-12-31
df_dates = pd.DataFrame(pd.date_range(start='2014-01-01', end='2025-12-31'), columns=['Date'])
df_dates.head()

,Date
0,2014-01-01
1,2014-01-02
2,2014-01-03
3,2014-01-04
4,2014-01-05


In [25]:
# 병합: 기존 US_dataset.ipynb와 동일한 left join 순서
df = pd.merge(df_dates, headlines, on='Date', how='left')
df = pd.merge(df, gold, on='Date', how='left')
df = pd.merge(df, oil, on='Date', how='left')
df = pd.merge(df, bond, on='Date', how='left')
df = pd.merge(df, cad, on='Date', how='left')
df = pd.merge(df, cny, on='Date', how='left')
df = pd.merge(df, eur, on='Date', how='left')
df = pd.merge(df, jpy, on='Date', how='left')
df = pd.merge(df, mxn, on='Date', how='left')
# Add VIX before ETF
df = pd.merge(df, vix, on='Date', how='left')
# ETF and Bitcoin
df = pd.merge(df, qqq, on='Date', how='left')
df = pd.merge(df, bitcoin, on='Date', how='left')
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4383 entries, 0 to 4382
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      4383 non-null   datetime64[ns]
 1   Headline  4382 non-null   object        
 2   gold      3075 non-null   float64       
 3   oil       3154 non-null   float64       
 4   bond      3130 non-null   float64       
 5   cad       3131 non-null   float64       
 6   cny       3132 non-null   float64       
 7   eur       3131 non-null   float64       
 8   jpy       3131 non-null   float64       
 9   mxn       3131 non-null   float64       
 10  vix       3057 non-null   float64       
 11  ETF       3019 non-null   float64       
 12  bitcoin   4383 non-null   float64       
dtypes: datetime64[ns](1), float64(11), object(1)
memory usage: 445.3+ KB
None


,Date,Headline,gold,oil,bond,cad,cny,eur,jpy,mxn,vix,ETF,bitcoin
0,2014-01-01,"As Latvia Adopts Euro, Future Growth Is Slowing",NaN,98.70,3.026,1.0641,6.0540,0.7271,105.23,13.0298,NaN,NaN,815.9
1,2014-01-02,Russell Banks: By the Book,1225.2,95.44,2.991,1.0668,6.0507,0.7314,104.76,13.1500,14.23,86.86,856.9
2,2014-01-03,Annual Sales for American Automakers Reach a 6...,1238.6,93.96,2.998,1.0632,6.0515,0.7359,104.82,13.0970,13.76,86.24,884.3
3,2014-01-04,"Help the Working Poor, but Share the Burden",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,924.7
4,2014-01-05,Democrats Press Revival of Jobless Aid That Ex...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1014.7


In [26]:
# 결측치 확인
df.isnull().sum()

Date           0
Headline       1
gold        1308
oil         1229
bond        1253
cad         1252
cny         1251
eur         1252
jpy         1252
mxn         1252
vix         1326
ETF         1364
bitcoin        0
dtype: int64

In [27]:
# Ensure ETF is the last column in output and include VIX
feature_cols = ['gold','oil','bond','cad','cny','eur','jpy','mxn','vix','bitcoin','ETF']
# Reorder columns
if 'Date' in df2.columns and 'Headline' in df2.columns:
    df2 = df2[['Date','Headline'] + [c for c in feature_cols if c in df2.columns]].copy()
# Clean headline text
df2['Headline'] = df2['Headline'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
# Save to CSV
out_path = BASE_DIR / 'research2.csv'
df2.to_csv(out_path, index=False)
# Show summary and preview
print(df2.info())
df2.head()

<class 'pandas.core.frame.DataFrame'>
Index: 4382 entries, 1 to 4382
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      4382 non-null   datetime64[ns]
 1   Headline  4382 non-null   object        
 2   gold      4382 non-null   float64       
 3   oil       4382 non-null   float64       
 4   bond      4382 non-null   float64       
 5   cad       4382 non-null   float64       
 6   cny       4382 non-null   float64       
 7   eur       4382 non-null   float64       
 8   jpy       4382 non-null   float64       
 9   mxn       4382 non-null   float64       
 10  vix       4382 non-null   float64       
 11  bitcoin   4382 non-null   float64       
 12  ETF       4382 non-null   float64       
dtypes: datetime64[ns](1), float64(11), object(1)
memory usage: 479.3+ KB
None


,Date,Headline,gold,oil,bond,cad,cny,eur,jpy,mxn,vix,bitcoin,ETF
1,2014-01-02,Russell Banks: By the Book,1225.2,95.44,2.991,1.0668,6.0507,0.7314,104.76,13.150,14.23,856.9,86.86
2,2014-01-03,Annual Sales for American Automakers Reach a 6...,1238.6,93.96,2.998,1.0632,6.0515,0.7359,104.82,13.097,13.76,884.3,86.24
3,2014-01-04,"Help the Working Poor, but Share the Burden",1238.6,93.96,2.998,1.0632,6.0515,0.7359,104.82,13.097,13.76,924.7,86.24
4,2014-01-05,Democrats Press Revival of Jobless Aid That Ex...,1238.6,93.96,2.998,1.0632,6.0515,0.7359,104.82,13.097,13.76,1014.7,86.24
5,2014-01-06,"Growth Returns to Tech, but Profits Will Not B...",1238.0,93.43,2.958,1.0653,6.0527,0.7338,104.21,13.076,13.55,1012.7,85.92


In [28]:
# CSV 저장 (Index, Date, Headline, 이후 특성 순서로 정렬)
output_path = BASE_DIR / 'research2.csv'
# 원하는 컬럼 순서 정의: ETF를 마지막에 배치하고 VIX 추가
feature_cols = ['gold','oil','bond','cad','cny','eur','jpy','mxn','vix','bitcoin','ETF']
cols = ['Date', 'Headline'] + feature_cols
# df2에 인덱스 컬럼 추가 (1부터 시작)
df_out = df2[cols].copy()

# 헤드라인 클렌징: 한국어/모지박(�, ?셲 등) 제거, 공백 정규화
import re

def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s)
    # U+FFFD replacement char 제거
    s = s.replace('\uFFFD', '')
    # 한글(완성형/자모) 제거
    s = re.sub(r'[\uAC00-\uD7A3\u1100-\u11FF\u3130-\u318F]', '', s)
    # 비 ASCII 문자 제거 (중국어/특수 모지박 포함)
    s = s.encode('ascii', 'ignore').decode('ascii')
    # 남은 ? 연속 제거 (모지박 패턴 흉터 제거)
    s = re.sub(r'\?+', ' ', s)
    # 공백 정규화
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_out['Headline'] = df_out['Headline'].apply(clean_text)

df_out.insert(0, 'Index', range(1, len(df_out) + 1))
df_out.to_csv(output_path, index=False)
output_path

WindowsPath('c:/Users/a00548169/OneDrive - ONEVIRTUALOFFICE/Desktop/자기주도적/ETFpricepredictmodel/data/2014-2025/USD/research2.csv')